# **Корреляционный анализ данных (Correlation Analysis)**

* __Цель корреляции:__ Изучить взаимосвязи между признаками внутри выделенных сегментов пациентов с помощью корреляционного анализа, научиться интерпретировать коэффициенты корреляции и выявлять мультиколлинеарность.
* __Задачи корреляции:__ Применить статистические методы для выявления силы и направления связей между переменными, оценить их значимость (p-value) и сравнить структуру корреляций в разных кластерах.
* __Алгоритм использования:__
  1. __Общая корреляционная матрица (Пирсон и Спирмен)__
  2. __Анализ статистической значимости корреляций (p-value)__
  3. __Проверка данных на мультиколлинеарность (VIF)__
  4. __Внутрикластерный корреляционный анализ (K-Means, DBSCAN, Иерархический)__

---

## **1. Общий корреляционный анализ**

* __Цель:__ Выявить и оценить базовые статистические взаимосвязи между всеми числовыми медицинскими признаками на уровне всей выборки пациентов (до разделения на специфические сегменты).
* __Задачи:__
  1. Рассчитать и визуально сравнить корреляционные матрицы Пирсона и Спирмена для определения природы связей (линейные или монотонные).
  2. Построить наглядные тепловые карты (Heatmaps) с отсечением дублирующейся информации для быстрого поиска наиболее сильных зависимостей.
  3. Вычислить статистическую значимость (p-value) каждого коэффициента для фильтрации случайного информационного шума и формирования достоверной матрицы статистически значимых корреляций.

In [ ]:
# --- ИМПОРТ БИБЛИОТЕК И ПЕРВИЧНАЯ НАСТРОЙКА ---

import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

from scripts.logger import setup_logger

# Инициализация логгера
logger = setup_logger(__name__)

# Настройка визуализации
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 8)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

In [ ]:
# --- 1. ЗАГРУЗКА ДАННЫХ И ПЕРВИЧНЫЙ АНАЛИЗ ---

try:
    data = pd.read_csv("../data/processed/heart_disease_uci_clustered.csv")
    display(Markdown("#### **Содержание набора данных с метками кластеров**"))
    display(data.head())
    display(
        Markdown(
            f"Размерность данных: **{data.shape[0]} строк и {data.shape[1]} столбцов.**"
        )
    )
except FileNotFoundError as e:
    logger.error(
        "Файл `../data/processed/heart_disease_uci_clustered.csv` не найден. Убедитесь, что пайплайн кластеризации был выполнен."
    )
    raise e

In [ ]:
# --- 2. ПОДГОТОВКА ДАННЫХ ДЛЯ АНАЛИЗА ---

display(Markdown("#### **Определение признаков для анализа**"))

# Определение признаков для анализа
cluster_cols = ["Cluster_KMeans", "Cluster_DBSCAN", "Cluster_Hierarchical"]
columns_to_exclude = set(cluster_cols + ["id", "num"])
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# Фильтрация признаков
feature_cols = [col for col in numeric_cols if col not in columns_to_exclude]

display(
    Markdown(
        f"* Отобрано **{len(feature_cols)}** числовых признаков для корреляционного анализа.\n\n"
        f"* **Признаки:** `{', '.join(feature_cols)}`."
    )
)

# Создание словарей с датасетами для внутрисегментного анализа
cluster_datasets = {}

for col in cluster_cols:
    algo_name = col.split("_")[1]

    cluster_datasets[algo_name] = {
        c_id: data[data[col] == c_id][feature_cols].copy()
        for c_id in sorted(data[col].dropna().unique())
    }

display(Markdown("#### **Структура разбиения данных по кластерам**"))
display(
    Markdown(
        f"* **K-Means:** {[int(k) for k in cluster_datasets['KMeans'].keys()]}\n\n"
        f"* **DBSCAN:** {[int(k) for k in cluster_datasets['DBSCAN'].keys()]} (где -1 - это шум)\n\n"
        f"* **Иерархическая:** {[int(k) for k in cluster_datasets['Hierarchical'].keys()]}"
    )
)

# Подготовка общего набора данных для анализа
display(Markdown("#### **Общий набор данных для анализа**"))
data_features = data[feature_cols].copy()
display(data_features.head())

In [ ]:
# --- 3. ОБЩАЯ КОРРЕЛЯЦИОННАЯ МАТРИЦА ---

display(Markdown("### **Общая корреляционная матрица (Пирсон и Спирмен)**"))

corr_pearson = data_features.corr(method="pearson")
corr_spearman = data_features.corr(method="spearman")

display(Markdown("* **Корреляционная матрица Пирсона (линейные связи):**"))
display(
    corr_pearson.style.background_gradient(cmap="coolwarm", axis=None).format(
        precision=3
    )
)

display(
    Markdown("* **Корреляционная матрица Спирмена (монотонные/нелинейные связи):**")
)
display(
    corr_spearman.style.background_gradient(cmap="coolwarm", axis=None).format(
        precision=3
    )
)

#### **Сравнительный анализ корреляционных матриц (Пирсон vs. Спирмен)**

* **Эмпирические наблюдения:** Сопоставление результатов вычисления коэффициентов линейной (Пирсон) и ранговой (Спирмен) корреляции демонстрирует высокую степень их согласованности. Абсолютная разность (дельта) значений для идентичных пар признаков минимальна и не превышает $0.02 - 0.04$. В частности, оценка взаимосвязи между возрастом (`age`) и максимальной ЧСС (`thalch`) по критерию Пирсона составляет $-0.351$, а по критерию Спирмена — $-0.329$.
* **Аналитическое заключение:** Подобная конвергенция (схождение) метрик статистически подтверждает **преимущественно линейную природу** взаимосвязей в исследуемом наборе данных. Кроме того, это свидетельствует об отсутствии значимых монотонных отклонений и экстремальных выбросов, к которым традиционно чувствителен линейный критерий. На основании этого, использование коэффициента корреляции Пирсона в качестве фундаментальной метрики для последующих этапов анализа является математически обоснованным.

In [ ]:
# --- 4. ВИЗУАЛИЗАЦИЯ КОРРЕЛЯЦИЙ ---

display(Markdown("### **Визуализация корреляционных матриц (Пирсон и Спирмен)**"))

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# Маски для скрытия верхнего треугольника
mask_pearson = np.triu(np.ones_like(corr_pearson, dtype=bool))
mask_spearman = np.triu(np.ones_like(corr_spearman, dtype=bool))

cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Визуализация корреляционной матрицы Пирсона с помощью тепловой карты (Рисунок 1)
sns.heatmap(
    corr_pearson,
    mask=mask_pearson,
    annot=True,
    fmt=".2f",
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=axes[0],
)
axes[0].set_title("Рисунок 1: Корреляция Пирсона (Линейная)", pad=15)
axes[0].grid(False)

# Визуализация корреляционной матрицы Спирмена с помощью тепловой карты (Рисунок 2)
sns.heatmap(
    corr_spearman,
    mask=mask_spearman,
    annot=True,
    fmt=".2f",
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=axes[1],
)
axes[1].set_title("Рисунок 2: Корреляция Спирмена (Ранговая / Монотонная)", pad=15)
axes[1].grid(False)

plt.tight_layout()
plt.show()

#### **Cтатистическая интерпретация выявленных паттернов**

* **Выраженная обратная корреляция: `age` (Возраст) и `thalch` (Максимальная ЧСС): $r = -0.35$**
  * *Клиническое обоснование:* Данная зависимость отражает фундаментальную физиологическую закономерность: с увеличением возраста максимальная частота сердечных сокращений, достигаемая на пике физической нагрузки, имеет естественную тенденцию к снижению. 
* **Умеренная прямая корреляция: `age` (Возраст) и `trestbps` (Артериальное давление в покое): $r = 0.24$**
  * *Клиническое обоснование:* Выявленная взаимосвязь объясняется возрастным снижением эластичности сосудистой стенки (преимущественно вследствие атеросклеротических процессов), что приводит к ожидаемому компенсаторному повышению базового артериального давления в старших возрастных когортах.
* **Умеренная прямая корреляция: `age` (Возраст) и `oldpeak` (Депрессия ST-сегмента): $r = 0.23$**
  * *Клиническое обоснование:* Статистически подтверждается, что с увеличением возраста возрастает вероятность фиксации ишемических изменений на ЭКГ (где депрессия ST является классическим маркером гипоксии миокарда) в ответ на физическую нагрузку.
* **Аналитическое наблюдение: Статистическая независимость признака `chol` (Уровень холестерина)**
  * *Интерпретация:* В масштабах всей выборки уровень холестерина демонстрирует отсутствие значимой линейной зависимости от остальных клинических показателей (коэффициенты корреляции стремятся к нулю). Отсутствие коллинеарности характеризует этот параметр как ортогональный (независимый) и высокоинформативный предиктор для построения будущих многомерных алгоритмов машинного обучения.

In [ ]:
# --- 5. АНАЛИЗ ЗНАЧИМОСТИ КОРРЕЛЯЦИЙ (P-VALUE) ---

display(Markdown("### **Анализ статистической значимости корреляций**"))

# Уровень значимости для тестирования гипотезы о нулевой корреляции
ALPHA_LEVEL = 0.05

# Векторизованное вычисление p-value
p_values = data_features.corr(method=lambda x, y: stats.pearsonr(x, y)[1])

# Заполнение диагональных элементов NaN безопасным сопособом через маску
diag_mask = np.eye(len(p_values), dtype=bool)
p_values = p_values.mask(diag_mask, np.nan)


# Функция для цветового форматирования p-value
def highlight_pvals(val) -> str:
    if pd.isna(val):
        return "color: gray"  # Диагональ
    elif val < ALPHA_LEVEL:
        return "color: green; font-weight: bold"  # Статистически значимые корреляции
    else:
        return "color: red"  # Случайные корреляции


display(Markdown(f"* **Матрица p-value (Уровень значимости α = {ALPHA_LEVEL})**"))
display(
    Markdown(
        f"*Зеленым цветом выделены статистически значимые значения (p < {ALPHA_LEVEL}), красным — случайные.*"
    )
)
display(p_values.style.map(highlight_pvals).format(precision=4, na_rep="—"))

# --- ФОРМИРОВАНИЕ МАТРИЦЫ ЗНАЧИМЫХ КОРРЕЛЯЦИЙ ---

significant_corr = corr_pearson.copy()
significant_corr = significant_corr.mask((p_values >= ALPHA_LEVEL) | (p_values.isna()))

display(Markdown("* **Матрица статистически значимых корреляций**"))
display(
    significant_corr.style.background_gradient(
        cmap="coolwarm", axis=None, vmin=-1, vmax=1
    )
    .highlight_null(color="#f4f4f4")
    .format(precision=3, na_rep="-")
)

#### **Оценка статистической значимости корреляционных взаимосвязей**

На данном этапе исследования была проведена статистическая проверка гипотез для оценки достоверности выявленных связей.
Нулевая гипотеза ($H_0$): *«Истинный коэффициент корреляции между признаками в генеральной совокупности равен нулю (связи нет)»*. Критический уровень статистической значимости (вероятность ошибки первого рода) был зафиксирован на стандартизированной отметке $\alpha = 0.05$.

##### **Ключевые результаты оценки (на основе расчета p-value):**

1. **Доказательство достоверности выраженных зависимостей:** Эмпирические паттерны, продемонстрировавшие высокие коэффициенты на этапе первичного анализа (в частности, обратная связь между возрастом `age` и максимальной ЧСС `thalch`, а также прямая связь между возрастом `age` и артериальным давлением `trestbps`), характеризовались значениями $p\text{-value} < 0.0001$. Это свидетельствует о том, что вероятность случайного возникновения подобных взаимосвязей в выборке пренебрежимо мала, что позволяет уверенно отвергнуть нулевую гипотезу ($H_0$) в отношении данных пар признаков.

2. **Нивелирование статистического шума (исключение ложноположительных связей):** Ряд слабовыраженных коэффициентов не преодолел заданный порог статистической значимости. Показательным примером является взаимосвязь между уровнем сывороточного холестерина (`chol`) и депрессией ST-сегмента (`oldpeak`), где расчетное значение $p\text{-value}$ превысило $0.05$. В контексте математической статистики подобные связи интерпретируются как случайные выборочные флуктуации и методически обоснованно исключаются из дальнейшего анализа.

3. **Аналитическая ценность результирующей матрицы:** Сформированная итоговая матрица значимых корреляций представляет собой отфильтрованный массив данных, очищенный от стохастического (случайного) шума. Она агрегирует исключительно робастные взаимосвязи, которые могут служить надежным и математически обоснованным фундаментом для обучения предиктивных алгоритмов машинного обучения, а также для формулирования объективных клинических заключений.

## **2. Анализ мультиколлинеарности данных**

* __Цель:__ Провести диагностику линейной зависимости между независимыми переменными (предикторами) для предотвращения проблемы мультиколлинеарности, которая способна вызвать дестабилизацию весовых коэффициентов в предиктивных алгоритмах.
* __Задачи:__
  1. Вычислить фактор инфляции дисперсии (Variance Inflation Factor, VIF) для каждого числового признака.
  2. Идентифицировать предикторы с критическим уровнем коллинеарности (VIF > 5 или VIF > 10).
  3. Сформировать оптимальную стратегию обработки выявленной мультиколлинеарности.

In [ ]:
# --- ДИАГНОСТИКА МУЛЬТИКОЛЛИНЕАРНОСТИ (VIF) ---

display(Markdown("#### **Расчет фактора инфляции дисперсии (VIF)**"))

# Пороговые значения для интерпретации VIF
VIF_SEVERE_THRESHOLD = 10
VIF_MODERATE_THRESHOLD = 5

# Добавление константы для корректного расчета VIF
X_with_const = sm.add_constant(data_features)

# Вычисление VIF для каждого признака с помощью генератора списков
vif_data = pd.DataFrame(
    {
        "Признак": X_with_const.columns,
        "VIF": [
            variance_inflation_factor(X_with_const.values, i)
            for i in range(X_with_const.shape[1])
        ],
    }
)

# Исключение фиктивной константы из итоговой таблицы и сортировка по убыванию VIF
vif_df = (
    vif_data[vif_data["Признак"] != "const"]
    .sort_values(by="VIF", ascending=False)
    .reset_index(drop=True)
)


# Функция для цветового форматирования значений VIF
def highlight_vif(val) -> str:
    if val >= VIF_SEVERE_THRESHOLD:
        return "color: red; font-weight: bold"  # Строгая мультиколлинеарность
    elif val >= VIF_MODERATE_THRESHOLD:
        return "color: orange; font-weight: bold"  # Умеренная мультиколлинеарность
    else:
        return "color: green"  # Безопасный уровень


# Вывод легенды и таблицы
display(
    Markdown(
        "* **VIF < 5:** Отсутствие значимой мультиколлинеарности (Зеленый)\n\n"
        "* **5 ≤ VIF < 10:** Умеренная мультиколлинеарность (Оранжевый)\n\n"
        "* **VIF ≥ 10:** Выраженная мультиколлинеарность (Красный)"
    )
)

display(Markdown("#### **Значения VIF для числовых предикторов**"))
display(vif_df.style.map(highlight_vif, subset=["VIF"]).format({"VIF": "{:.3f}"}))

#### **Аналитическое заключение по результатам диагностики VIF**

* **Интерпретация полученных метрик:** По результатам расчета Variance Inflation Factor (VIF), значения для всех исследуемых клинических предикторов находятся в абсолютно безопасном диапазоне (от 1.06 до 1.23). Ни один из признаков не приближается к критическому порогу (VIF ≥ 5.0).
* **Диагноз:** Проблема мультиколлинеарности в анализируемом наборе данных **отсутствует**. Каждый признак вносит свой уникальный, статистически независимый вклад в дисперсию общего признакового пространства.
* **Стратегия дальнейшей обработки:** Поскольку выраженной линейной зависимости между предикторами не выявлено (признаки не дублируют друг друга), исключение переменных или применение методов снижения размерности (например, PCA) на данном этапе является нецелесообразным. 
* **Итог:** Весь набор из 5 числовых предикторов (`age`, `thalch`, `trestbps`, `oldpeak`, `chol`) признан валидным и может быть использован для обучения предиктивных алгоритмов без риска дестабилизации их весовых коэффициентов.

## **3. Внутрикластерный корреляционный анализ**

* __Цель:__ Исследовать локальные корреляционные структуры внутри каждого выделенного сегмента пациентов и выявить скрытые паттерны, которые нивелируются при глобальном анализе всей выборки.
* __Задачи:__
  1. Построить изолированные корреляционные матрицы для каждого кластера в рамках алгоритмов K-Means, DBSCAN и Иерархической кластеризации.
  2. Визуализировать матрицы в виде тепловых карт.
  3. Провести сравнительный анализ: зафиксировать усиление/ослабление связей, появление новых (локальных) корреляций и исчезновение глобальных трендов.

In [ ]:
# --- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---

# Пороговые значения для интерпретации силы корреляции
CORR_STRONG_THRESHOLD = 0.4
CORR_MODERATE_THRESHOLD = 0.25


# --- ФУНКЦИЯ ВИЗУАЛИЗАЦИИ (ТЕПЛОВЫЕ КАРТЫ) ---


def plot_cluster_heatmaps(
    clusters_dict: dict[int, pd.DataFrame], algo_name: str, start_fig_num: int
) -> None:
    """
    Строит тепловые карты корреляций для набора кластеров.
    """

    n_clusters = len(clusters_dict)
    fig, axes = plt.subplots(1, n_clusters, figsize=(6 * n_clusters, 6))
    if n_clusters == 1:
        axes = [axes]

    cmap = sns.diverging_palette(230, 20, as_cmap=True)

    for idx, (c_id, df_c) in enumerate(clusters_dict.items()):
        corr_matrix = df_c.corr(method="pearson")
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

        sns.heatmap(
            corr_matrix,
            mask=mask,
            annot=True,
            fmt=".2f",
            cmap=cmap,
            vmin=-1,
            vmax=1,
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8},
            ax=axes[idx],
        )

        axes[idx].set_title(
            f"Рисунок {start_fig_num + idx}: {algo_name} (Кластер {c_id})\n$N$ = {len(df_c)}",
            pad=15,
        )
        axes[idx].grid(False)

    plt.tight_layout()
    plt.show()


# --- ФУНКЦИЯ АГРЕГАЦИИ ДАННЫХ ---


def build_comparison_dataframe(
    clusters_dict: dict[int, pd.DataFrame], features: list
) -> pd.DataFrame:
    """
    Строит DataFrame для сравнения корреляций между кластерами.
    """

    pairs = list(itertools.combinations(features, 2))
    comparison_data = {f"{p[0]} — {p[1]}": [] for p in pairs}
    cluster_labels = [f"Кластер {c_id}" for c_id in clusters_dict.keys()]

    for df_c in clusters_dict.values():
        corr_matrix = df_c.corr(method="pearson")

        for p in pairs:
            val = corr_matrix.loc[p[0], p[1]]
            comparison_data[f"{p[0]} — {p[1]}"].append(val)

    return pd.DataFrame(comparison_data, index=cluster_labels).T


# --- ФУНКЦИЯ СТИЛИЗАЦИИ ---


def highlight_comparison(val) -> str:
    """Цветовое форматирование таблицы сравнения корреляций между кластерами."""

    if pd.isna(val):
        return "color: gray"
    if abs(val) >= CORR_STRONG_THRESHOLD:
        return "color: red; font-weight: bold"
    if abs(val) >= CORR_MODERATE_THRESHOLD:
        return "color: orange; font-weight: bold"

    return ""

In [ ]:
# --- 1. АНАЛИЗ K-MEANS ---

display(Markdown("### **Внутрикластерный анализ: K-Means**"))
display(
    Markdown(
        "* **Тепловая карта (Heatmap) корреляций для каждого кластера, выделенного K-Means.**"
    )
)
plot_cluster_heatmaps(cluster_datasets["KMeans"], "K-Means", start_fig_num=3)

comp_df_kmeans = build_comparison_dataframe(cluster_datasets["KMeans"], feature_cols)
display(Markdown("* **Сравнение корреляций (K-Means)**"))
display(comp_df_kmeans.style.map(highlight_comparison).format(precision=3, na_rep="—"))


# --- 2. АНАЛИЗ DBSCAN ---

display(Markdown("### **Внутрикластерный анализ: DBSCAN**"))
display(
    Markdown(
        "* **Тепловая карта (Heatmap) корреляций для каждого кластера, выделенного DBSCAN.**\n\n"
        "*Примечание: Шумовые точки (Кластер -1) исключены из визуализации для оценки чистых сегментов.*"
    )
)

# Исключение шумовых точек (-1) перед передачей в функции визуализации и анализа
dbscan_clean = {k: v for k, v in cluster_datasets["DBSCAN"].items() if k != -1}
plot_cluster_heatmaps(dbscan_clean, "DBSCAN", start_fig_num=6)

comp_df_dbscan = build_comparison_dataframe(cluster_datasets["DBSCAN"], feature_cols)
display(Markdown("* **Сравнение корреляций (DBSCAN)**"))
display(comp_df_dbscan.style.map(highlight_comparison).format(precision=3, na_rep="—"))


# --- 3. АНАЛИЗ ИЕРАРХИЧЕСКОЙ КЛАСТЕРИЗАЦИИ ---

display(Markdown("### **Внутрикластерный анализ: Иерархическая кластеризация**"))
display(
    Markdown(
        "* **Тепловая карта (Heatmap) корреляций для каждого кластера, выделенного иерархической кластеризацией.**"
    )
)
plot_cluster_heatmaps(
    cluster_datasets["Hierarchical"], "Иерархическая", start_fig_num=9
)

comp_df_hier = build_comparison_dataframe(
    cluster_datasets["Hierarchical"], feature_cols
)
display(Markdown("* **Сравнение корреляций (Иерархическая)**"))
display(comp_df_hier.style.map(highlight_comparison).format(precision=3, na_rep="—"))

#### **Анализ и интерпретация внутрикластерных корреляций**

**1. Анализ сегментации K-Means:**

* **Гомогенизация связей:** В обоих кластерах K-Means отсутствуют сильные или умеренные корреляции (все значения $|r| < 0.25$). 
* **Интерпретация:** Алгоритм K-Means разделил пространство таким образом, что максимизировал межкластерную дисперсию и минимизировал внутрикластерную. Пациенты внутри сегментов настолько похожи друг на друга по базовым характеристикам, что выраженные линейные зависимости внутри самих групп практически исчезли.

**2. Анализ сегментации DBSCAN:**

* **Хаотичность шума (Кластер -1):** Подавляющее большинство корреляций в шумовом сегменте стремится к нулю. Это подтверждает, что DBSCAN математически корректно отделил неструктурированные аномалии от плотных ядер.
* **Фундаментальные физиологические законы:** Единственная значимая связь в шумовом кластере — это `age` — `thalch` ($r = -0.394$). Значит, снижение максимальной частоты сердечных сокращений с возрастом является настолько строгим физиологическим законом, что он сохраняется даже у пациентов-outliers, чьи остальные показатели абсолютно аномальны.
* **Специфика ядер (Кластеры 0 и 1):** Очищенные от шума сегменты показывают свои уникальные паттерны. Основное ядро (Кластер 0) усиливает классические тренды (например, `age` — `oldpeak`, $r = 0.340$), а узкий Кластер 1 демонстрирует уникальную сильную обратную связь `trestbps` — `chol` ($r = -0.508$).
* **Нулевая дисперсия:** Появление прочерков (`—` / `NaN`) в Кластере 1 свидетельствует о том, что алгоритм нашел настолько плотную группу (core points), что по некоторым признакам (например, `age` или `oldpeak`) пациенты в ней практически идентичны (дисперсия равна нулю, из-за чего корреляцию математически вычислить невозможно).

**3. Анализ Иерархической кластеризации:**

* **Локальное усиление связи (Кластер 2):** В данном сегменте зафиксировано заметное усиление прямой зависимости между возрастом и артериальным давлением (`age` — `trestbps`, $r = 0.339$) по сравнению с глобальной метрикой ($r = 0.241$). 
* **Интерпретация:** Иерархический алгоритм успешно изолировал когорту пациентов, для которых фактор "возрастной сосудистой жесткости" является доминирующим и проявляется наиболее агрессивно. В Кластерах 1 и 3 эта зависимость выражена значительно слабее.

## **4. Сравнительный анализ методов кластеризации**

* __Цель:__ Провести комплексную оценку и сравнение эффективности алгоритмов машинного обучения (K-Means, DBSCAN, Иерархическая кластеризация) с точки зрения их способности выделять структурно однородные и клинически интерпретируемые сегменты пациентов.
* __Задачи:__
  1. Рассчитать количественные метрики внутрикластерной однородности для каждого выделенного сегмента, включая анализ шумовых выбросов.
  2. Провести визуальное сопоставление силы внутренних связей между различными алгоритмами.
  3. Сформировать клинико-статистические профили (интерпретации) для ключевых групп пациентов на основе локальных корреляционных паттернов.

In [ ]:
# --- СРАВНИТЕЛЬНЫЙ АНАЛИЗ МЕТОДОВ ---

display(Markdown("### **Сравнительный анализ методов кластеризации**"))

# --- 1. РАСЧЕТ МЕТРИК ОДНОРОДНОСТИ ---


def calculate_cluster_metrics(algo_name: str, clusters_dict: dict) -> list:
    """
    Вычисляет метрики однородности для каждого кластера, включая шум DBSCAN.
    """

    metrics = []
    for c_id, df_c in clusters_dict.items():
        corr_matrix = df_c.corr().abs()

        # Вычисление средней абсолютной корреляции (исключая диагональ)
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        mean_corr = corr_matrix.where(mask).stack().mean()

        metrics.append(
            {
                "Алгоритм": algo_name,
                "Тип": "Шум" if c_id == -1 else "Кластер",
                "Сегмент": c_id,
                "Размер (N)": len(df_c),
                "Средняя сила связи (|r|)": mean_corr,
            }
        )
    return metrics


# Сбор метрик для всех алгоритмов
all_metrics = []
all_metrics.extend(calculate_cluster_metrics("K-Means", cluster_datasets["KMeans"]))
all_metrics.extend(calculate_cluster_metrics("DBSCAN", cluster_datasets["DBSCAN"]))
all_metrics.extend(
    calculate_cluster_metrics("Hierarchical", cluster_datasets["Hierarchical"])
)

metrics_df = pd.DataFrame(all_metrics)

display(
    Markdown(
        "* **Сводные метрики однородности кластерных структур**\n\n"
        "*Метрика 'Средняя сила связи' показывает, насколько сильно признаки связаны друг с другом внутри сегмента.*"
    )
)
display(
    metrics_df.style.background_gradient(
        subset=["Средняя сила связи (|r|)"], cmap="YlGn"
    ).format(precision=3)
)

In [ ]:
# --- 2. ВИЗУАЛИЗАЦИЯ ---


display(Markdown("* **Сравнение средней силы корреляций внутри кластеров**"))

plt.figure(figsize=(14, 7))

# Создание вспомогательной колонки для красивой легенды графика
metrics_df["Метка"] = metrics_df["Тип"] + " " + metrics_df["Сегмент"].astype(str)

ax = sns.barplot(
    data=metrics_df,
    x="Алгоритм",
    y="Средняя сила связи (|r|)",
    hue="Метка",
    palette="Set3",
    edgecolor="black",
)

# Оформление графика
plt.title(
    "Сравнение средней силы корреляций внутри кластеров по методам", pad=20, fontsize=14
)
plt.ylabel("Средняя абсолютная корреляция (|r|)", fontsize=12)
plt.xlabel("Алгоритм кластеризации", fontsize=12)

plt.legend(title="Сегмент", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Добавление текстовых подписей значений над каждым столбцом
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height) and height > 0:
        ax.annotate(
            f"{height:.3f}",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            xytext=(0, 5),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold",
            color="#333333",
        )

plt.tight_layout()
plt.show()

# Удаление вспомогательной колонки после визуализации
metrics_df = metrics_df.drop(columns=["Метка"])

#### **Сравнение методов кластеризации через призму корреляций**

* **Иллюзия гомогенности (K-Means).** Алгоритм успешно справился с задачей минимизации внутрикластерной дисперсии. Однако с клинической точки зрения это привело к «схлопыванию» метрик: пациенты внутри сегментов 0 и 1 оказались настолько похожи друг на друга, что любые выраженные линейные тренды ($|r| < 0.25$) перестали прослеживаться. 
* **Фундаментальные законы и ложные аномалии (DBSCAN).** Алгоритм блестяще изолировал информационный шум (Кластер -1), доказав, что даже у пациентов с аномальными характеристиками сохраняется жесткий физиологический закон: снижение максимального пульса с возрастом ($r = -0.394$). Одновременно с этим DBSCAN обнаружил сверхплотный микросегмент (Кластер 1) с парадоксальной обратной связью между давлением и холестерином ($r = -0.508$). Однако из-за критически малого размера выборки ($N=11$) данный инсайт признан статистически неустойчивым (риск overfitting'а).
* **Золотая середина (Иерархическая кластеризация).** Метод продемонстрировал наибольшую устойчивость. В отличие от K-Means, он сохранил дисперсию признаков, а в отличие от DBSCAN — избежал дробления на микро-группы. Это позволило достоверно выявить сегмент пациентов (Кластер 2, $N=177$) с наиболее агрессивным проявлением возрастной гипертензии ($r = 0.339$).

##### **Итоговая рекомендация**

Для задач поиска скрытых клинических взаимосвязей оптимальным признается использование **иерархической кластеризации**. Алгоритм обеспечивает наилучший баланс между математической изоляцией групп и сохранением статистической значимости найденных трендов.